# Thesis — Runtime Benchmark

Measures real per-epoch wall time for the dual-ResNet50 model on **this machine**, for the
dataset picked by the `DATASET_DIR` switch, then extrapolates the full `thesis_final.ipynb`
run (Optuna search + final retrain). Run it once per machine to get a measured estimate
instead of a guess.

It does **not** train to convergence — it times a handful of train/val steps and scales up.
Takes ~1–2 minutes.

### 1 — Setup + dataset toggle
Imports, device, seed, and the same dataset selector as `thesis_final.ipynb`. Set the
toggle to the dataset this machine will actually run.

In [1]:
import os, time, math, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')
SEED = 42; torch.manual_seed(SEED); np.random.seed(SEED)
device = torch.device('mps' if torch.backends.mps.is_available() else
                      'cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device} | PyTorch {torch.__version__}')

def sync():
    if device.type == 'cuda': torch.cuda.synchronize()
    elif device.type == 'mps': torch.mps.synchronize()

Device: mps | PyTorch 2.4.1


In [2]:
# ── DATASET SELECTOR (match what this machine will run in thesis_final) ──
DATASET_DIR  = Path('dataset') / 'Milk10k'    # or Path('dataset') / 'Derm7pt'
DATASET_NAME = DATASET_DIR.name.lower()
assert DATASET_NAME in ('derm7pt', 'milk10k')
print(f'>>> {DATASET_NAME}')

>>> milk10k


### 2 — Build loaders
Identical data pipeline to `thesis_final.ipynb` (paired images, 70/15/15 split, batch 64,
`drop_last=True`), so the measured batch counts match the real run.

In [3]:
if DATASET_NAME == 'derm7pt':
    raw = pd.read_csv(DATASET_DIR / 'meta' / 'meta.csv')
    raw['clinic_path'] = raw['clinic'].apply(lambda x: str(DATASET_DIR / 'images' / x))
    raw['derm_path']   = raw['derm'].apply(lambda x: str(DATASET_DIR / 'images' / x))
    raw['diagnosis']   = raw['diagnosis'].str.strip().str.lower()
    grp = {'melanoma':'MEL','melanoma (less than 0.76 mm)':'MEL','melanoma (in situ)':'MEL',
           'melanoma (0.76 to 1.5 mm)':'MEL','melanoma (more than 1.5 mm)':'MEL','melanoma metastasis':'MEL',
           'clark nevus':'NV','reed or spitz nevus':'NV','dermal nevus':'NV','blue nevus':'NV',
           'congenital nevus':'NV','combined nevus':'NV','recurrent nevus':'NV',
           'basal cell carcinoma':'BCC','seborrheic keratosis':'SK','lentigo':'MISC',
           'dermatofibroma':'MISC','vascular lesion':'MISC','melanosis':'MISC','miscellaneous':'MISC'}
    raw['diagnosis_group'] = raw['diagnosis'].map(grp)
    dataset_df = raw[['clinic_path','derm_path','diagnosis_group']].copy()
else:
    meta = pd.read_csv(DATASET_DIR / 'MILK10k_Training_Metadata.csv')
    gt   = pd.read_csv(DATASET_DIR / 'MILK10k_Training_GroundTruth.csv')
    cmap = {'MEL':'MEL','NV':'NV','BCC':'BCC','BKL':'SK','DF':'MISC','VASC':'MISC'}
    drop = {'AKIEC','SCCKA','INF','BEN_OTH','MAL_OTH'}
    cols = [c for c in gt.columns if c != 'lesion_id']
    gt['raw_class'] = gt[cols].idxmax(axis=1)
    gt = gt[~gt['raw_class'].isin(drop)].copy()
    gt['diagnosis_group'] = gt['raw_class'].map(cmap)
    cm = meta[meta['image_type']=='clinical: close-up'][['lesion_id','isic_id']].rename(columns={'isic_id':'clinic_isic'})
    dm = meta[meta['image_type']=='dermoscopic'][['lesion_id','isic_id']].rename(columns={'isic_id':'derm_isic'})
    p = cm.merge(dm, on='lesion_id')
    r = gt[['lesion_id','diagnosis_group']].merge(p, on='lesion_id')
    r['clinic_path'] = r.apply(lambda x: str(DATASET_DIR/'MILK10k_Training_Input'/x['lesion_id']/f"{x['clinic_isic']}.jpg"), axis=1)
    r['derm_path']   = r.apply(lambda x: str(DATASET_DIR/'MILK10k_Training_Input'/x['lesion_id']/f"{x['derm_isic']}.jpg"), axis=1)
    dataset_df = r[['clinic_path','derm_path','diagnosis_group']].copy()

class_names = ['BCC','MEL','MISC','NV','SK']
label_map = {n:i for i,n in enumerate(class_names)}
dataset_df['label'] = dataset_df['diagnosis_group'].map(label_map)
dataset_df = dataset_df.dropna(subset=['label']).reset_index(drop=True)
dataset_df['label'] = dataset_df['label'].astype(int)

labels = dataset_df['label'].values
idx = np.arange(len(dataset_df))
tr, tmp = train_test_split(idx, test_size=0.30, stratify=labels, random_state=SEED)
va, te  = train_test_split(tmp, test_size=0.50, stratify=labels[tmp], random_state=SEED)
BATCH_SIZE = 64
N_TRAIN, N_VAL = len(tr), len(va)
BATCHES_TRAIN = N_TRAIN // BATCH_SIZE              # drop_last=True
BATCHES_VAL   = math.ceil(N_VAL / BATCH_SIZE)
print(f'{DATASET_NAME}: train={N_TRAIN} val={N_VAL} | train batches/epoch={BATCHES_TRAIN} val batches={BATCHES_VAL}')

class DS(Dataset):
    def __init__(self, df, tf): self.df=df.reset_index(drop=True); self.tf=tf
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r=self.df.iloc[i]
        c=Image.open(r['clinic_path']).convert('RGB'); d=Image.open(r['derm_path']).convert('RGB')
        return self.tf(c), self.tf(d), torch.tensor(r['label'])
tf_train = transforms.Compose([transforms.Resize((256,256)), transforms.RandomResizedCrop(224,scale=(0.8,1.0)),
    transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(), transforms.RandomRotation(20),
    transforms.ColorJitter(0.2,0.2,0.2,0.1), transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
tf_val = transforms.Compose([transforms.Resize((256,256)), transforms.CenterCrop(224), transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
NW = 0 if device.type in ('mps','cpu') else min(8, os.cpu_count() or 4)  # Windows/CUDA gets workers
train_loader = DataLoader(DS(dataset_df.iloc[tr], tf_train), batch_size=BATCH_SIZE, shuffle=True,
                          drop_last=True, num_workers=NW, pin_memory=(device.type=='cuda'))
val_loader   = DataLoader(DS(dataset_df.iloc[va], tf_val), batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NW, pin_memory=(device.type=='cuda'))

milk10k: train=3052 val=654 | train batches/epoch=47 val batches=11


### 3 — Model
Same dual-ResNet50 (`mul` fusion is representative; `concat` adds a tiny 1×1 conv, `add` is
identical cost). `pretrained=False` — skips the weight download; forward/backward speed is
the same either way.

In [4]:
class ResNet50Backbone(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        b = models.resnet50(weights=models.ResNet50_Weights.DEFAULT if pretrained else None)
        self.features = nn.Sequential(b.conv1,b.bn1,b.relu,b.maxpool,b.layer1,b.layer2,b.layer3,b.layer4)
    def forward(self,x): return self.features(x)

class DualBranch(nn.Module):
    def __init__(self, fusion='mul', dropout=0.32):
        super().__init__()
        self.rc = ResNet50Backbone(False); self.rd = ResNet50Backbone(False)
        self.post = nn.Sequential(nn.BatchNorm2d(2048), nn.ReLU(True))
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(nn.Linear(2048,512), nn.BatchNorm1d(512), nn.ReLU(True),
                                nn.Dropout(dropout), nn.Linear(512,5))
    def forward(self,c,d):
        fc=self.rc(c); fd=self.rd(d)
        x=self.pool(self.post(fc*fd)).flatten(1)
        return self.fc(x)

model = DualBranch().to(device)
opt = optim.AdamW(model.parameters(), lr=7.5e-4)
crit = nn.CrossEntropyLoss()
print('Model ready.')

Model ready.


### 4 — Time train + val steps
Warm up a few steps (first steps include lazy init / cudnn autotune), then time train
(forward+backward+step) and val (forward only) batches.

In [5]:
WARMUP, MEASURE = 3, 12
it = iter(train_loader)
def next_batch():
    global it
    try: return next(it)
    except StopIteration:
        it = iter(train_loader); return next(it)

# ── train step timing ──
model.train()
for _ in range(WARMUP):
    c,d,y = next_batch(); c,d,y = c.to(device),d.to(device),y.to(device)
    opt.zero_grad(); loss = crit(model(c,d), y); loss.backward(); opt.step()
sync(); t0 = time.time()
for _ in range(MEASURE):
    c,d,y = next_batch(); c,d,y = c.to(device),d.to(device),y.to(device)
    opt.zero_grad(); loss = crit(model(c,d), y); loss.backward(); opt.step()
sync(); sec_train_batch = (time.time()-t0)/MEASURE

# ── val step timing (forward only) ──
model.eval()
vit = iter(val_loader)
def next_val():
    global vit
    try: return next(vit)
    except StopIteration:
        vit = iter(val_loader); return next(vit)
with torch.no_grad():
    for _ in range(WARMUP):
        c,d,y = next_val(); model(c.to(device), d.to(device))
    sync(); t0 = time.time()
    for _ in range(MEASURE):
        c,d,y = next_val(); model(c.to(device), d.to(device))
    sync(); sec_val_batch = (time.time()-t0)/MEASURE

print(f'train: {sec_train_batch*1000:6.0f} ms/batch  ({BATCH_SIZE/sec_train_batch:5.0f} img/s)')
print(f'val  : {sec_val_batch*1000:6.0f} ms/batch  ({BATCH_SIZE/sec_val_batch:5.0f} img/s)')

train:   2505 ms/batch  (   26 img/s)
val  :    598 ms/batch  (  107 img/s)


### 5 — Extrapolate the full `thesis_final` run
Per-epoch cost = 1 train pass + 2 val passes (`train_trial` runs `validate_improved` **and**
`val_f1_macro` each epoch). Search = 35 trials × 28 epochs, shown both without pruning
(worst case) and with MedianPruner (~16 effective epochs/trial). Final retrain = 60 epochs.
Contrastive pretrain (~30 epochs, paid once) shown separately — leave it out if you drop the
toggle.

In [6]:
epoch_train = BATCHES_TRAIN * sec_train_batch
epoch_val   = BATCHES_VAL   * sec_val_batch
epoch_cost  = epoch_train + 2*epoch_val          # train_trial does 2 val passes/epoch
print(f'~ {epoch_cost:5.1f} s / epoch  (train {epoch_train:.1f}s + 2x val {epoch_val:.1f}s)\n')

N_TRIALS, SEARCH_EPOCHS, FINAL_EPOCHS = 35, 28, 60
PRUNED_AVG_EPOCHS = 16     # rough MedianPruner effective average
CONTRASTIVE_EPOCHS = 30    # one-time, train-pass cost only (no val)

def hrs(s): return s/3600
search_worst = N_TRIALS * SEARCH_EPOCHS    * epoch_cost
search_prune = N_TRIALS * PRUNED_AVG_EPOCHS* epoch_cost
final_cost   = FINAL_EPOCHS * epoch_cost
contrastive  = CONTRASTIVE_EPOCHS * epoch_train

print(f'{"phase":<34}{"hours":>8}')
print(f'{"search (no pruning, worst case)":<34}{hrs(search_worst):>8.1f}')
print(f'{"search (with MedianPruner ~est)":<34}{hrs(search_prune):>8.1f}')
print(f'{"final retrain (60 ep)":<34}{hrs(final_cost):>8.1f}')
print(f'{"contrastive pretrain (one-time)":<34}{hrs(contrastive):>8.1f}')
print('-'*42)
lo = hrs(search_prune + final_cost)
hi = hrs(search_worst + final_cost)
print(f'{"TOTAL (no contrastive)":<34}{lo:>7.1f}-{hi:.1f} h')
print(f'{"TOTAL (+ contrastive)":<34}{lo+hrs(contrastive):>7.1f}-{hi+hrs(contrastive):.1f} h')

~ 130.9 s / epoch  (train 117.7s + 2x val 6.6s)

phase                                hours
search (no pruning, worst case)       35.6
search (with MedianPruner ~est)       20.4
final retrain (60 ep)                  2.2
contrastive pretrain (one-time)        1.0
------------------------------------------
TOTAL (no contrastive)               22.5-37.8 h
TOTAL (+ contrastive)                23.5-38.8 h


In [7]:
# ── Scale estimate to the OTHER dataset (compute-bound: scales with train batches) ──
OTHER = {'derm7pt': 707, 'milk10k': 3000}   # approx train sizes (paired)
other_name = 'derm7pt' if DATASET_NAME=='milk10k' else 'milk10k'
ratio = (OTHER[other_name]//BATCH_SIZE) / max(1, BATCHES_TRAIN)
print(f'This run measured on {DATASET_NAME} ({BATCHES_TRAIN} train batches/epoch).')
print(f'Rough scale to {other_name} (~{OTHER[other_name]} train, ratio x{ratio:.2f}) — '
      f'multiply the totals above by ~{ratio:.2f}.')
print('NOTE: assumes the other dataset is compute-bound the same way; tiny sets (Derm7pt) '
      'are often data-loading-bound, so its real time can be higher than the raw ratio.')

This run measured on milk10k (47 train batches/epoch).
Rough scale to derm7pt (~707 train, ratio x0.23) — multiply the totals above by ~0.23.
NOTE: assumes the other dataset is compute-bound the same way; tiny sets (Derm7pt) are often data-loading-bound, so its real time can be higher than the raw ratio.
